# 🔧 Week 2: Feature Engineering & Data Pipeline
## Telco Customer Churn — Building a Production-Ready ML Pipeline

**Goal:** Transform raw data into a clean, feature-rich dataset and wrap it all in a reusable sklearn `Pipeline`.

### Agenda
1. Load & validate the raw data
2. Inspect engineered features
3. Analyse feature distributions & correlations with churn
4. Run the full sklearn pipeline
5. Save processed arrays for Week 3 (modelling)

---
## 0. Setup

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on the path
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from src.data.load_data import load_raw
from src.data.features import engineer_all_features
from src.data.preprocess import drop_ids, handle_missing, encode_target
from src.data.pipeline import build_pipeline, prepare_dataframe, NUMERICAL_FEATURES, CATEGORICAL_FEATURES
from sklearn.model_selection import train_test_split

sns.set_theme(style='darkgrid', palette='deep')
plt.rcParams['figure.dpi'] = 120
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.3f}'.format)

print('Setup complete ✓')

---
## 1. Load Raw Data

In [ ]:
df_raw = load_raw()
print(f'Shape: {df_raw.shape}')
df_raw.head(3)

In [ ]:
# Quick sanity check
print('Dtypes:')
print(df_raw.dtypes)
print(f'\nMissing values: {df_raw.isnull().sum().sum()}')

---
## 2. Apply Preprocessing + Feature Engineering

In [ ]:
df = df_raw.copy()
df = handle_missing(df)
df = engineer_all_features(df)
df = encode_target(df, target_col='Churn')

print(f'Columns after engineering: {df.shape[1]}')
new_cols = ['clv', 'avg_monthly_charge', 'charge_increase',
            'contract_stability', 'service_bundle_score',
            'has_internet', 'tenure_band', 'is_high_value']
df[new_cols + ['Churn']].describe(include='all')

---
## 3. Feature Analysis

### 3a. CLV Distribution by Churn

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col in zip(axes, ['clv', 'avg_monthly_charge', 'charge_increase']):
    sns.kdeplot(data=df, x=col, hue='Churn', ax=ax, fill=True, alpha=0.4)
    ax.set_title(f'{col} distribution')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.suptitle('CLV-derived features by Churn', y=1.02, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 3b. Churn Rate by Tenure Band

In [ ]:
band_order = ['new', 'mid', 'loyal', 'champion']
churn_by_band = (
    df.groupby('tenure_band')['Churn']
    .mean()
    .reindex(band_order)
    .mul(100)
    .round(1)
)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(churn_by_band.index, churn_by_band.values,
               color=sns.color_palette('rocket', 4))
ax.bar_label(bars, fmt='%.1f%%', padding=3)
ax.set_ylabel('Churn Rate (%)')
ax.set_xlabel('Tenure Band')
ax.set_title('Churn Rate by Customer Tenure Band', fontweight='bold')
ax.set_ylim(0, churn_by_band.max() * 1.2)
plt.tight_layout()
plt.show()

print(churn_by_band.to_string())

### 3c. Service Bundle Score vs Churn

In [ ]:
bundle_churn = (
    df.groupby('service_bundle_score')['Churn']
    .agg(['mean', 'count'])
    .rename(columns={'mean': 'churn_rate', 'count': 'n_customers'})
    .mul({'churn_rate': 100, 'n_customers': 1})
    .round({'churn_rate': 1})
)

fig, ax1 = plt.subplots(figsize=(10, 4))
ax2 = ax1.twinx()

ax1.bar(bundle_churn.index, bundle_churn['n_customers'],
         alpha=0.4, color='steelblue', label='# Customers')
ax2.plot(bundle_churn.index, bundle_churn['churn_rate'],
          color='crimson', marker='o', linewidth=2, label='Churn Rate %')

ax1.set_xlabel('Service Bundle Score (# add-ons)')
ax1.set_ylabel('# Customers', color='steelblue')
ax2.set_ylabel('Churn Rate (%)', color='crimson')
ax1.set_title('Churn Rate by Service Bundle Score', fontweight='bold')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.tight_layout()
plt.show()

print(bundle_churn)

### 3d. Correlation Heatmap — Numerical Features

In [ ]:
num_feat_cols = ['tenure', 'MonthlyCharges', 'TotalCharges',
                 'clv', 'avg_monthly_charge', 'charge_increase',
                 'contract_stability', 'service_bundle_score',
                 'has_internet', 'is_high_value', 'Churn']

corr = df[num_feat_cols].corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
    center=0, linewidths=0.5, ax=ax,
    annot_kws={'size': 8}
)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Run the Full sklearn Pipeline

In [ ]:
X, y = prepare_dataframe(df_raw)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

num_cols = [c for c in NUMERICAL_FEATURES if c in X.columns]
cat_cols = [c for c in CATEGORICAL_FEATURES if c in X.columns]

pipe = build_pipeline(numerical_features=num_cols, categorical_features=cat_cols)

X_train_t = pipe.fit_transform(X_train)
X_test_t  = pipe.transform(X_test)

print(f'Train set: {X_train_t.shape}')
print(f'Test set:  {X_test_t.shape}')
print(f'Churn rate train: {y_train.mean():.3f}')
print(f'Churn rate test:  {y_test.mean():.3f}')

---
## 5. Save Processed Data for Week 3

In [ ]:
import numpy as np

PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(exist_ok=True)

np.save(PROCESSED_DIR / 'X_train.npy', X_train_t)
np.save(PROCESSED_DIR / 'X_test.npy',  X_test_t)
np.save(PROCESSED_DIR / 'y_train.npy', y_train.values)
np.save(PROCESSED_DIR / 'y_test.npy',  y_test.values)

print('Saved to data/processed/:')
for f in PROCESSED_DIR.glob('*.npy'):
    print(f'  {f.name}  ({f.stat().st_size / 1024:.1f} KB)')

---
## ✅ Week 2 Summary

| Feature | Description | Churn Signal |
|---------|-------------|--------------|
| `clv` | tenure × MonthlyCharges | High CLV → lower churn |
| `avg_monthly_charge` | TotalCharges / tenure | Stable spenders less likely to churn |
| `charge_increase` | Recent vs historical spend | Sudden increases → higher risk |
| `contract_stability` | 0/1/2 ordinal for contract type | Month-to-month = highest churn |
| `service_bundle_score` | # add-ons subscribed | More services → more sticky |
| `has_internet` | Any internet service | Internet users churn differently |
| `tenure_band` | new / mid / loyal / champion | New customers churn most |
| `is_high_value` | CLV ≥ 75th percentile | Valuable customers worth targeted retention |

**Next: Week 3 — Model Development & Experiment Tracking with MLflow**